In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import os, time, csv, shutil
import pandas as pd
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.metrics import f1_score
import random
import time
from torch.utils.data import TensorDataset
from torchvision.transforms import v2
from torch.utils.data.dataloader import default_collate

In [2]:
cinic_mean_RGB = [0.47889522, 0.47227842, 0.43047404]
cinic_std_RGB  = [0.24205776, 0.23828046, 0.25874835]

SEEDS      = [42, 123, 2024, 7, 999]
NUM_EPOCHS = 25
DATA_PATH  = '/kaggle/input/datasets/mengcius/cinic10/'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

Using device: cuda


# Ensuring reproducibility

In [3]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Data loaders

In [4]:
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
])

image_aug_pipelines = {
    'flip': transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'blur': transforms.Compose([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'jitter': transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutout': transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.25, 0.25), ratio=(1, 1), value=0),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha1': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha4': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
}

batch_aug_pipelines = {
    'cutmix_alpha1': transforms.v2.CutMix(num_classes=10, alpha=1.0),
    'cutmix_alpha4': transforms.v2.CutMix(num_classes=10, alpha=4.0),
}

def preload_to_ram(dataset):
    loader = DataLoader(dataset, batch_size=512, num_workers=4, pin_memory=False)
    all_images, all_labels = [], []
    for images, labels in loader:
        all_images.append(images)
        all_labels.append(labels)
    return TensorDataset(torch.cat(all_images), torch.cat(all_labels))

def get_cutmix_collate_fn(aug_type, p=0.5):
    def collate_fn(batch):
        cutmix = batch_aug_pipelines[aug_type]
        images, labels = default_collate(batch)
        if torch.rand(()) < p:
            images, labels = cutmix(images, labels)
        return images, labels
    return collate_fn

In [5]:
_loader_cache = {}

def get_loaders(batch_size: int, path: str = DATA_PATH, aug_type: str = None):
    train_base = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=base_transform)
    valid_ds   = datasets.ImageFolder(root=os.path.join(path, 'valid'), transform=base_transform)
    test_ds    = datasets.ImageFolder(root=os.path.join(path, 'test'),  transform=base_transform)

    key = (batch_size, aug_type)
    if key in _loader_cache:
        return _loader_cache[key]

    valid_ds = preload_to_ram(valid_ds)
    test_ds  = preload_to_ram(test_ds)

    if aug_type is None:
        train_base = preload_to_ram(train_base)
        train_ds = train_base
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=4, pin_memory=True, persistent_workers=True)
    else:
        if aug_type in ('cutmix_alpha1', 'cutmix_alpha4'):
            chosen_transform = image_aug_pipelines[aug_type]
            train_aug = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=chosen_transform)
            train_ds  = ConcatDataset([train_base, train_aug])
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                      num_workers=4, pin_memory=True, persistent_workers=True,
                                      collate_fn=get_cutmix_collate_fn(aug_type))
        else:
            if aug_type == 'random':
                standard_transforms = {k: v for k, v in image_aug_pipelines.items() 
                                       if k not in ('cutmix_alpha1', 'cutmix_alpha4')}
                chosen_transform = transforms.RandomChoice(list(standard_transforms.values()))
            else:
                chosen_transform = image_aug_pipelines[aug_type]
            train_aug = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=chosen_transform)
            train_ds  = ConcatDataset([train_base, train_aug])
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                      num_workers=4, pin_memory=True, persistent_workers=True)

    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)

    _loader_cache[key] = (train_loader, valid_loader, test_loader)
    return _loader_cache[key]

I created my own custom cnn architecture that consists of 4 convolutional blocks (each block looks like this: Conv → BN → ReLU → Conv → BN → ReLU → MaxPool).

Number of channels increases as follows with each block: 64 → 128 → 256 → 256.

Moreover, the network uses residual skip connection in blocks 3 & 4, dropout before final classifier (in baseline this value is set to 0)

In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, use_residual: bool = False):
        super().__init__()
        self.use_residual = use_residual

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        ) if use_residual and in_ch != out_ch else nn.Identity()

        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        out = self.block(x)
        if self.use_residual:
            out = out + self.skip(x)
        out = self.relu(out)
        out = self.pool(out)
        return out

In [7]:
class CustomCNN(nn.Module):
    """
    Architecture:
        Block 1: 3   → 32  (no residual)   32×32 → 16×16
        Block 2: 32  → 64  (no residual)   16×16 → 8×8
        Block 3: 64  → 128 (residual)      8×8   → 4×4
        Block 4: 128 → 256 (residual)      4×4   → 2×2
        GAP → Dropout → FC(256, 10)

    """
    def __init__(self, num_classes: int = 10, dropout: float = 0.0):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(3,   32,  use_residual=False),
            ConvBlock(32,  64,  use_residual=False),
            ConvBlock(64,  128, use_residual=True),
            ConvBlock(128, 256, use_residual=True),
        )

        self.gap        = nn.AdaptiveAvgPool2d(1)
        self.dropout    = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(256, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick check

In [8]:
_test_model = CustomCNN()
_test_input = torch.randn(2, 3, 32, 32)
_test_out   = _test_model(_test_input)
assert _test_out.shape == (2, 10), "Output shape mismatch!"
print(f"CustomCNN parameters: {count_params(_test_model):,}")
print(f"Output shape: {_test_out.shape}")
del _test_model, _test_input, _test_out

CustomCNN parameters: 1,217,514
Output shape: torch.Size([2, 10])


## Training

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        if labels.dim() == 2: 
            all_labels.extend(labels.argmax(dim=1).cpu().numpy())
        else:
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1

In [10]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1

In [11]:
def run_experiment(
    config_name: str,
    lr: float,
    batch_size: int,
    seeds: list = SEEDS,
    num_epochs: int = NUM_EPOCHS,
    dropout: float = 0.0,
    weight_decay: float = 0.0,
    aug_type: str = None,
):
    """
    Runs a single hyperparameter configuration across all seeds.
    Returns a dict with mean ± std of validation and test F1.
    """
    print(f"\n{'='*60}")
    print(f"  Config: {config_name}")
    print(f"  LR={lr}  BS={batch_size}  dropout={dropout}  wd={weight_decay}")
    print(f"{'='*60}")

    val_f1_per_seed, test_f1_per_seed = [], []
    histories = []

    for seed in seeds:
        set_seed(seed)
        print(f"\n  ── Seed {seed} ──")

        train_loader, valid_loader, test_loader = get_loaders(batch_size, aug_type=aug_type)

        model     = CustomCNN(num_classes=10, dropout=dropout).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        best_val_f1 = 0.0
        best_state  = None
        history     = []

        for epoch in range(1, num_epochs + 1):
            t0 = time.time()
            train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
            val_loss,   val_f1   = evaluate(model, valid_loader, criterion, DEVICE)
            elapsed = time.time() - t0

            history.append({
                'epoch': epoch, 'seed': seed, 'config': config_name,
                'train_loss': train_loss, 'train_f1': train_f1,
                'val_loss': val_loss,     'val_f1': val_f1,
            })

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            if epoch % 5 == 0 or epoch == num_epochs:
                print(f"    Epoch {epoch:2d}/{num_epochs} | "
                      f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | "
                      f"({elapsed:.1f}s)")

        model.load_state_dict(best_state)
        _, test_f1 = evaluate(model, test_loader, criterion, DEVICE)
        print(f"  → Best Val F1: {best_val_f1:.4f}  |  Test F1: {test_f1:.4f}")

        val_f1_per_seed.append(best_val_f1)
        test_f1_per_seed.append(test_f1)
        histories.extend(history)

    result = {
        'config':        config_name,
        'lr':            lr,
        'batch_size':    batch_size,
        'dropout':       dropout,
        'weight_decay':  weight_decay,
        'val_f1_mean':   np.mean(val_f1_per_seed),
        'val_f1_std':    np.std(val_f1_per_seed),
        'test_f1_mean':  np.mean(test_f1_per_seed),
        'test_f1_std':   np.std(test_f1_per_seed),
        'val_f1_seeds':  val_f1_per_seed,
        'test_f1_seeds': test_f1_per_seed,
    }

    print(f"\n Val  F1: {result['val_f1_mean']:.4f} ± {result['val_f1_std']:.4f}")
    print(f" Test F1: {result['test_f1_mean']:.4f} ± {result['test_f1_std']:.4f}")

    return result, pd.DataFrame(histories)

# Learning rate testing

In [ ]:
BASELINE_BS = 64
LR_VALUES   = [1e-2, 1e-3, 1e-4]

phase1_results   = []
phase1_histories = []

for lr in LR_VALUES:
    cfg_name = f"LR={lr:.0e}_BS={BASELINE_BS}"
    res, hist = run_experiment(cfg_name, lr=lr, batch_size=BASELINE_BS)
    phase1_results.append(res)
    phase1_histories.append(hist)

phase1_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase1_results])

print("\n\n" + "─"*60)
print("PHASE 1 RESULTS — Learning Rate Sweep (BS=64)")
print("─"*60)
print(phase1_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

In [ ]:
phase1_df.to_csv('custom_cnn_lr_results.csv', index=False)
phase1_hist_df = pd.concat(phase1_histories, ignore_index=True)
phase1_hist_df.to_csv('custom_cnn_lr_history.csv', index=False)

# Batch size testing

In [ ]:
best_lr_idx = phase1_df['val_f1_mean'].idxmax()
best_lr     = phase1_df.loc[best_lr_idx, 'lr']

In [ ]:
BS_VALUES = [32, 64, 128]

phase2_results   = []
phase2_histories = []

for bs in BS_VALUES:
    cfg_name = f"LR={best_lr:.0e}_BS={bs}"
    res, hist = run_experiment(cfg_name, lr=best_lr, batch_size=bs)
    phase2_results.append(res)
    phase2_histories.append(hist)

phase2_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase2_results])

print("\n\n" + "─"*60)
print(f"PHASE 2 RESULTS — Batch Size Sweep (LR={best_lr:.0e})")
print("─"*60)
print(phase2_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))



## Saving results

In [ ]:
phase2_df.to_csv('custom_cnn_bs_results.csv', index=False)
phase2_hist_df = pd.concat(phase2_histories, ignore_index=True)
phase2_hist_df.to_csv('custom_cnn_bs_history.csv', index=False)

# Dropout rate testing

In [ ]:
BEST_LR = 1e-2
BEST_BS = 64
DROPOUT_VALUES = [0.0, 0.2, 0.3]

phase3_results   = []
phase3_histories = []

for dropout in DROPOUT_VALUES:
    cfg_name = f"LR={BEST_LR:.0e}_BS={BEST_BS}_dropout={dropout}"
    res, hist = run_experiment(cfg_name, lr=BEST_LR, batch_size=BEST_BS, dropout=dropout)
    phase3_results.append(res)
    phase3_histories.append(hist)

phase3_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase3_results])

print("\n\n" + "─"*60)
print("PHASE 3 RESULTS — Dropout Sweep")
print("─"*60)
print(phase3_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

best_dropout_idx = phase3_df['val_f1_mean'].idxmax()
best_dropout     = phase3_df.loc[best_dropout_idx, 'dropout']
print(f"\n Best dropout from Phase 3: {best_dropout}")

phase3_df.to_csv('custom_cnn_phase3_results.csv', index=False)
phase3_hist_df = pd.concat(phase3_histories, ignore_index=True)
phase3_hist_df.to_csv('custom_cnn_phase3_history.csv', index=False)

# Weight decay testing

In [ ]:
WD_VALUES = [0.0, 1e-5, 1e-4]

phase4_results   = []
phase4_histories = []

for wd in WD_VALUES:
    cfg_name = f"LR={BEST_LR:.0e}_BS={BEST_BS}_dropout={best_dropout}_wd={wd}"
    res, hist = run_experiment(cfg_name, lr=BEST_LR, batch_size=BEST_BS, dropout=best_dropout, weight_decay=wd)
    phase4_results.append(res)
    phase4_histories.append(hist)

phase4_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase4_results])

print("\n\n" + "─"*60)
print("PHASE 4 RESULTS — Weight Decay Sweep")
print("─"*60)
print(phase4_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

best_wd_idx = phase4_df['val_f1_mean'].idxmax()
best_wd     = phase4_df.loc[best_wd_idx, 'weight_decay']
print(f"\n Best weight decay from Phase 4: {best_wd}")

phase4_df.to_csv('custom_cnn_phase4_results.csv', index=False)
phase4_hist_df = pd.concat(phase4_histories, ignore_index=True)
phase4_hist_df.to_csv('custom_cnn_phase4_history.csv', index=False)

print(f"\n Best overall config:")
print(f"  LR={BEST_LR:.0e} | BS={BEST_BS} | dropout={best_dropout} | weight_decay={best_wd}")

# Augmentation sweep

In [ ]:
BEST_LR      = 1e-2
BEST_BS      = 64
BEST_DROPOUT = 0.3
BEST_WD      = 0.0

AUG_VALUES = [None, 'flip', 'blur', 'jitter']

phase5_results   = []
phase5_histories = []

for aug in AUG_VALUES:
    aug_label = aug if aug is not None else 'none'
    cfg_name  = f"LR={BEST_LR:.0e}_BS={BEST_BS}_dropout={BEST_DROPOUT}_wd={BEST_WD}_aug={aug_label}"
    res, hist = run_experiment(
        cfg_name,
        lr=BEST_LR,
        batch_size=BEST_BS,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WD,
        aug_type=aug,
    )
    phase5_results.append(res)
    phase5_histories.append(hist)

phase5_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase5_results])

print("\n\n" + "─"*60)
print("PHASE 5 RESULTS — Augmentation Sweep")
print("─"*60)
print(phase5_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

best_aug_idx = phase5_df['val_f1_mean'].idxmax()
best_aug     = phase5_df.loc[best_aug_idx, 'config']
print(f"\n Best aug from Phase 5: {best_aug}")

phase5_df.to_csv('custom_cnn_phase5_results.csv', index=False)
phase5_hist_df = pd.concat(phase5_histories, ignore_index=True)
phase5_hist_df.to_csv('custom_cnn_phase5_history.csv', index=False)

In [ ]:
BEST_LR      = 1e-2
BEST_BS      = 64
BEST_DROPOUT = 0.3
BEST_WD      = 0.0

AUG_VALUES = ['cutmix_alpha1', 'cutmix_alpha4']

phase6_results   = []
phase6_histories = []

for aug in AUG_VALUES:
    cfg_name = f"LR={BEST_LR:.0e}_BS={BEST_BS}_dropout={BEST_DROPOUT}_wd={BEST_WD}_aug={aug}"
    res, hist = run_experiment(
        cfg_name,
        lr=BEST_LR,
        batch_size=BEST_BS,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WD,
        aug_type=aug,
    )
    phase6_results.append(res)
    phase6_histories.append(hist)

phase6_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase6_results])

print("\n\n" + "─"*60)
print("PHASE 6 RESULTS — Advanced Augmentation Sweep")
print("─"*60)
print(phase6_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

phase6_df.to_csv('custom_cnn_phase6_results.csv', index=False)
phase6_hist_df = pd.concat(phase6_histories, ignore_index=True)
phase6_hist_df.to_csv('custom_cnn_phase6_history.csv', index=False)

In [ ]:
BEST_LR      = 1e-2
BEST_BS      = 64
BEST_DROPOUT = 0.3
BEST_WD      = 0.0

AUG_VALUES = ['cutout', 'random']

phase7_results   = []
phase7_histories = []

for aug in AUG_VALUES:
    cfg_name = f"LR={BEST_LR:.0e}_BS={BEST_BS}_dropout={BEST_DROPOUT}_wd={BEST_WD}_aug={aug}"
    res, hist = run_experiment(
        cfg_name,
        lr=BEST_LR,
        batch_size=BEST_BS,
        dropout=BEST_DROPOUT,
        weight_decay=BEST_WD,
        aug_type=aug,
    )
    phase7_results.append(res)
    phase7_histories.append(hist)

phase7_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase7_results])

print("\n\n" + "─"*60)
print("PHASE 7 RESULTS — CutOut & Random Augmentation")
print("─"*60)
print(phase7_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

phase7_df.to_csv('custom_cnn_phase7_results.csv', index=False)
phase7_hist_df = pd.concat(phase7_histories, ignore_index=True)
phase7_hist_df.to_csv('custom_cnn_phase7_history.csv', index=False)

Few shot learning

In [ ]:
from torch.utils.data import Subset
import random

def get_few_shot_loader(n_per_class: int, batch_size: int, path: str = DATA_PATH, seed: int = 42):
    """
    Tworzy loader z n_per_class próbkami per klasa z zbioru treningowego.
    """
    set_seed(seed)
    full_train = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=base_transform)
    
    # Grupuj indeksy per klasa
    class_indices = {}
    for idx, (_, label) in enumerate(full_train.samples):
        if label not in class_indices:
            class_indices[label] = []
        class_indices[label].append(idx)
    
    # Losuj n_per_class z każdej klasy
    selected_indices = []
    for label in sorted(class_indices.keys()):
        indices = class_indices[label]
        selected = random.sample(indices, n_per_class)
        selected_indices.extend(selected)
    
    few_shot_ds = Subset(full_train, selected_indices)
    few_shot_ds = preload_to_ram(few_shot_ds)
    
    return DataLoader(few_shot_ds, batch_size=batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, persistent_workers=True)


# ─────────────────────────────────────────────
# PHASE 8: Few-Shot Learning
# ─────────────────────────────────────────────
BEST_LR      = 1e-2
BEST_BS      = 64
BEST_DROPOUT = 0.3
BEST_WD      = 0.0
N_SHOTS      = [5, 10, 20]

phase8_results   = []
phase8_histories = []

for n in N_SHOTS:
    print(f"\n{'='*60}")
    print(f"  Few-Shot: {n} samples per class ({n*10} total)")
    print(f"{'='*60}")

    val_f1_per_seed, test_f1_per_seed = [], []
    histories = []

    _, valid_loader, test_loader = get_loaders(BEST_BS)

    for seed in SEEDS:
        set_seed(seed)
        print(f"\n  ── Seed {seed} ──")

        train_loader = get_few_shot_loader(n_per_class=n, batch_size=BEST_BS, seed=seed)

        model     = CustomCNN(num_classes=10, dropout=BEST_DROPOUT).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=BEST_LR, weight_decay=BEST_WD)

        best_val_f1 = 0.0
        best_state  = None
        history     = []

        for epoch in range(1, NUM_EPOCHS + 1):
            t0 = time.time()
            train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
            val_loss,   val_f1   = evaluate(model, valid_loader, criterion, DEVICE)
            elapsed = time.time() - t0

            history.append({
                'epoch': epoch, 'seed': seed, 'n_shots': n,
                'train_loss': train_loss, 'train_f1': train_f1,
                'val_loss': val_loss,     'val_f1': val_f1,
            })

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            if epoch % 5 == 0 or epoch == NUM_EPOCHS:
                print(f"    Epoch {epoch:2d}/{NUM_EPOCHS} | "
                      f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | ({elapsed:.1f}s)")

        model.load_state_dict(best_state)
        _, test_f1 = evaluate(model, test_loader, criterion, DEVICE)
        print(f"   Best Val F1: {best_val_f1:.4f}  |  Test F1: {test_f1:.4f}")

        val_f1_per_seed.append(best_val_f1)
        test_f1_per_seed.append(test_f1)
        histories.extend(history)

    result = {
        'n_shots':       n,
        'total_samples': n * 10,
        'val_f1_mean':   np.mean(val_f1_per_seed),
        'val_f1_std':    np.std(val_f1_per_seed),
        'test_f1_mean':  np.mean(test_f1_per_seed),
        'test_f1_std':   np.std(test_f1_per_seed),
    }

    print(f"\n   Val  F1: {result['val_f1_mean']:.4f} ± {result['val_f1_std']:.4f}")
    print(f"   Test F1: {result['test_f1_mean']:.4f} ± {result['test_f1_std']:.4f}")

    phase8_results.append(result)
    phase8_histories.append(pd.DataFrame(histories))

phase8_df = pd.DataFrame(phase8_results)

print("\n\n" + "─"*60)
print("PHASE 8 RESULTS — Few-Shot Learning")
print("─"*60)
print(phase8_df.to_string(index=False))

phase8_df.to_csv('custom_cnn_phase8_fewshot_results.csv', index=False)
phase8_hist_df = pd.concat(phase8_histories, ignore_index=True)
phase8_hist_df.to_csv('custom_cnn_phase8_fewshot_history.csv', index=False)

# Explainability

In [23]:
model = CustomCNN(num_classes=10, dropout=0.3)
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        print(name)

features.0.block.0
features.0.block.3
features.1.block.0
features.1.block.3
features.2.block.0
features.2.block.3
features.2.skip.0
features.3.block.0
features.3.block.3
features.3.skip.0


In [24]:
def train_and_save_best_model(
    lr: float = 1e-2,
    batch_size: int = 64,
    dropout: float = 0.3,
    weight_decay: float = 0.0,
    seeds: list = SEEDS,
    num_epochs: int = NUM_EPOCHS,
    aug_type: str = None,
    save_path: str = 'best_custom_cnn.pth',
):
    print("Training model for XAI analysis...")
    print(f"LR={lr} | BS={batch_size} | dropout={dropout} | wd={weight_decay}")

    best_val_f1_overall = 0.0
    best_state_overall  = None

    for seed in seeds:
        set_seed(seed)
        print(f"\n── Seed {seed} ──")

        train_loader, valid_loader, _ = get_loaders(batch_size, aug_type=aug_type)

        model     = CustomCNN(num_classes=10, dropout=dropout).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        best_val_f1 = 0.0
        best_state  = None

        for epoch in range(1, num_epochs + 1):
            train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
            _, val_f1 = evaluate(model, valid_loader, criterion, DEVICE)

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            if epoch % 5 == 0 or epoch == num_epochs:
                print(f"  Epoch {epoch:2d}/{num_epochs} | Val F1: {val_f1:.4f}")

        print(f" Best Val F1 for seed {seed}: {best_val_f1:.4f}")

        seed_save_path = save_path.replace('.pth', f'_seed_{seed}.pth')
        torch.save(best_state, seed_save_path)

        if best_val_f1 > best_val_f1_overall:
            best_val_f1_overall = best_val_f1
            best_state_overall  = best_state
            print(f"   New best overall model!")

    torch.save(best_state_overall, save_path)
    print(f"\n Best model saved to '{save_path}'")
    print(f"  Best Val F1: {best_val_f1_overall:.4f}")

train_and_save_best_model(aug_type='flip')

Training model for XAI analysis...
LR=0.01 | BS=64 | dropout=0.3 | wd=0.0

── Seed 42 ──
  Epoch  5/25 | Val F1: 0.7213
  Epoch 10/25 | Val F1: 0.7473
  Epoch 15/25 | Val F1: 0.7566
  Epoch 20/25 | Val F1: 0.7560
  Epoch 25/25 | Val F1: 0.7572
 Best Val F1 for seed 42: 0.7586
   New best overall model!

── Seed 123 ──
  Epoch  5/25 | Val F1: 0.7295
  Epoch 10/25 | Val F1: 0.7503
  Epoch 15/25 | Val F1: 0.7499
  Epoch 20/25 | Val F1: 0.7550
  Epoch 25/25 | Val F1: 0.7518
 Best Val F1 for seed 123: 0.7558

── Seed 2024 ──
  Epoch  5/25 | Val F1: 0.7311
  Epoch 10/25 | Val F1: 0.7493
  Epoch 15/25 | Val F1: 0.7493
  Epoch 20/25 | Val F1: 0.7470
  Epoch 25/25 | Val F1: 0.7519
 Best Val F1 for seed 2024: 0.7551

── Seed 7 ──
  Epoch  5/25 | Val F1: 0.7327
  Epoch 10/25 | Val F1: 0.7452
  Epoch 15/25 | Val F1: 0.7521
  Epoch 20/25 | Val F1: 0.7502
  Epoch 25/25 | Val F1: 0.7584
 Best Val F1 for seed 7: 0.7584

── Seed 999 ──
  Epoch  5/25 | Val F1: 0.7229
  Epoch 10/25 | Val F1: 0.7534
  Epo